In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

In [2]:
VALID_RANGES = {
    'sg':             {'min': 1.000, 'max': 1.035, 'label': 'Specific Gravity (sg) [e.g. 1.020]'},
    'al':             {'min': 0.0,   'max': 5.0,   'label': 'Albumin level (al) [0.0 to 5.0]'},
    'bgr':            {'min': 50.0,  'max': 500.0, 'label': 'Blood Glucose Random (bgr) [mg/dL]'},
    'sc':             {'min': 0.1,   'max': 15.0,  'label': 'Serum Creatinine (sc) [mg/dL]'},
    'hemo':           {'min': 3.0,   'max': 20.0,  'label': 'Hemoglobin (hemo) [g/dL]'},
    'rc':             {'min': 1.0,   'max': 8.0,   'label': 'Red Blood Cell Count (rc) [mill/mcl]'},
    'htn':            {'min': 0.0,   'max': 1.0,   'label': 'Hypertension history (htn) [0=No, 1=Yes]'},
    'rbc_is_missing': {'min': 0.0,   'max': 1.0,   'label': 'RBC entry missing status [0=No, 1=Yes]'}
}

In [5]:
print("==================================================")
print("🩺 CHRONIC KIDNEY DISEASE DIAGNOSTIC TERMINAL")
print("==================================================")
print("Please provide the patient details below:\n")

client_inputs = {}
is_valid_entry = True
violating_features = []

for feature, bounds in VALID_RANGES.items():
    try:
        # Request live entry from the client via terminal
        user_raw = input(f"👉 Enter {bounds['label']}: ")
        val = float(user_raw)
        
        # Verify if value is within normal/acceptable physiological range
        if val < bounds['min'] or val > bounds['max']:
            is_valid_entry = False
            violating_features.append(f"{feature} (Value: {val} out of range {bounds['min']}-{bounds['max']})")
        
        client_inputs[feature] = val
        
    except ValueError:
        print("🚨 Invalid entry format! You must type a numeric value.")
        is_valid_entry = False
        break

# Handle final range validation failure if any outliers were encountered
if not is_valid_entry and len(violating_features) > 0:
    print("\n🚨 Invalid Input")
    print("-" * 30)
    for error in violating_features:
        print(f"Boundary Violation Check Failed: {error}")
    print("Execution halted. Please re-run and input correct baseline thresholds.")

🩺 CHRONIC KIDNEY DISEASE DIAGNOSTIC TERMINAL
Please provide the patient details below:



👉 Enter Specific Gravity (sg) [e.g. 1.020]:  1.020
👉 Enter Albumin level (al) [0.0 to 5.0]:  0.0
👉 Enter Blood Glucose Random (bgr) [mg/dL]:  110
👉 Enter Serum Creatinine (sc) [mg/dL]:  1
👉 Enter Hemoglobin (hemo) [g/dL]:  15
👉 Enter Red Blood Cell Count (rc) [mill/mcl]:  5
👉 Enter Hypertension history (htn) [0=No, 1=Yes]:  0
👉 Enter RBC entry missing status [0=No, 1=Yes]:  0


In [6]:
# Execute only if the client inputs passed all boundary range filters
if is_valid_entry:
    try:
        # Step A: Import your saved production scaler and best model
        loaded_scaler = joblib.load('scaler.joblib')
        loaded_model = joblib.load('final_ckd_model.joblib')
        
        # Step B: Format the valid client inputs into a single-row Pandas DataFrame
        feature_names = ['sg', 'al', 'bgr', 'sc', 'hemo', 'rc', 'htn', 'rbc_is_missing']
        input_df = pd.DataFrame([client_inputs], columns=feature_names)
        
        # Step C: Use the imported production scaler to scale the fresh inputs
        input_scaled = pd.DataFrame(
            loaded_scaler.transform(input_df),
            columns=feature_names
        )
        
        # Step D: Feed the scaled input matrix into the deployed model for prediction
        prediction = loaded_model.predict(input_scaled)[0]
        probability = loaded_model.predict_proba(input_scaled)[0][1]
        
        # Step E: Determine and inform the client whether they are sick or not
        print("\n📋 Diagnostic Conclusion")
        print("==================================================")
        if prediction == 1:
            print(f"🔴 Result: Patient is Sick (Chronic Kidney Disease Confirmed)")
            print(f"The model calculated a {probability * 100:.2f}% operational risk probability.")
        else:
            print(f"🟢 Result: Patient is Healthy (No Sign of Chronic Kidney Disease)")
            print(f"The model calculated a remaining residual risk probability of only {probability * 100:.2f}%.")
        print("==================================================")
            
    except FileNotFoundError:
        print("\n🚨 Missing System Files! Ensure 'scaler.joblib' and 'final_ckd_model.joblib' are in your current directory.")
    except Exception as e:
        print(f"\nAn unexpected operational exception occurred during prediction processing: {e}")


📋 Diagnostic Conclusion
🟢 Result: Patient is Healthy (No Sign of Chronic Kidney Disease)
The model calculated a remaining residual risk probability of only 0.11%.


In [8]:
# =========================================================================
# 📋 INTERACTIVE CLIENT INPUT, IMMEDIATE VALIDATION & INSTANT PREDICTION
# =========================================================================
print("==================================================")
print("🩺 CHRONIC KIDNEY DISEASE DIAGNOSTIC TERMINAL")
print("==================================================")
print("Please provide the patient details below:\n")

client_inputs = {}
is_valid_entry = True
violating_features = []

# Loop through all 8 variables to prompt the client
for feature, bounds in VALID_RANGES.items():
    try:
        # Request live entry from the client via terminal
        user_raw = input(f"👉 Enter {bounds['label']}: ")
        val = float(user_raw)
        
        # Verify if value is within normal/acceptable physiological range
        if val < bounds['min'] or val > bounds['max']:
            is_valid_entry = False
            violating_features.append(f"{feature} (Value: {val} out of range {bounds['min']}-{bounds['max']})")
        
        client_inputs[feature] = val
        
    except ValueError:
        print("🚨 Invalid entry format! You must type a numeric value.")
        is_valid_entry = False
        break

# --- INSTANT EXECUTION GATEWAY ---
# As soon as the loop ends (after the 8th enter key press), this executes immediately.

if not is_valid_entry and len(violating_features) > 0:
    print("\n🚨 Invalid Input")
    print("-" * 30)
    for error in violating_features:
        print(f"Boundary Violation Check Failed: {error}")
    print("Execution halted. Please re-run and input correct baseline thresholds.")

elif is_valid_entry:
    try:
        # Step A: Instantly import your saved production scaler and best model
        loaded_scaler = joblib.load('scaler.joblib')
        loaded_model = joblib.load('final_ckd_model.joblib')
        
        # Step B: Format the valid client inputs into a single-row Pandas DataFrame
        feature_names = ['sg', 'al', 'bgr', 'sc', 'hemo', 'rc', 'htn', 'rbc_is_missing']
        input_df = pd.DataFrame([client_inputs], columns=feature_names)
        
        # Step C: Use the imported production scaler to scale the fresh inputs
        input_scaled = pd.DataFrame(
            loaded_scaler.transform(input_df),
            columns=feature_names
        )
        
        # Step D: Feed the scaled input matrix into the deployed model for prediction
        prediction = loaded_model.predict(input_scaled)[0]
        probability = loaded_model.predict_proba(input_scaled)[0][1]
        
        # Step E: Determine and immediately inform the client whether they are sick or not
        print("\n📋 Diagnostic Conclusion")
        print("==================================================")
        if prediction == 1:
            print(f"🔴 Result: Patient is Sick (Chronic Kidney Disease Confirmed)")
            print(f"The model calculated a {probability * 100:.2f}% operational risk probability.")
        else:
            print(f"🟢 Result: Patient is Healthy (No Sign of Chronic Kidney Disease)")
            print(f"The model calculated a remaining residual risk probability of only {probability * 100:.2f}%.")
        print("==================================================")
            
    except FileNotFoundError:
        print("\n🚨 Missing System Files! Ensure 'scaler.joblib' and 'final_ckd_model.joblib' are in your current directory.")
    except Exception as e:
        print(f"\nAn unexpected operational exception occurred during prediction processing: {e}")

🩺 CHRONIC KIDNEY DISEASE DIAGNOSTIC TERMINAL
Please provide the patient details below:



👉 Enter Specific Gravity (sg) [e.g. 1.020]:  1.01
👉 Enter Albumin level (al) [0.0 to 5.0]:  4
👉 Enter Blood Glucose Random (bgr) [mg/dL]:  280
👉 Enter Serum Creatinine (sc) [mg/dL]:  6.8
👉 Enter Hemoglobin (hemo) [g/dL]:  8.5
👉 Enter Red Blood Cell Count (rc) [mill/mcl]:  2.9
👉 Enter Hypertension history (htn) [0=No, 1=Yes]:  1
👉 Enter RBC entry missing status [0=No, 1=Yes]:  0



📋 Diagnostic Conclusion
🔴 Result: Patient is Sick (Chronic Kidney Disease Confirmed)
The model calculated a 100.00% operational risk probability.
